In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, Input
from tensorflow.keras.models import Sequential
import keras_tuner as kt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

In [2]:
df = pd.read_csv('dataset.csv')
df.head()

,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 246945 entries, 0 to 246944
Columns: 378 entries, diseases to neck weakness
dtypes: int64(377), str(1)
memory usage: 712.2 MB


In [4]:
# 2. Encode Disease Names to Integers
label_encoder = LabelEncoder()
df['disease_label'] = label_encoder.fit_transform(df.iloc[:, 0]) 

# Fix fragmentation by creating a consolidated copy
df = df.copy()

C:\Users\Bhagyashree\AppData\Local\Temp\ipykernel_19424\1922540394.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['disease_label'] = label_encoder.fit_transform(df.iloc[:, 0])


In [5]:
# Save the encoder for the Web App to use later
joblib.dump(label_encoder, 'disease_label_encoder.pkl')

['disease_label_encoder.pkl']

In [6]:
# 3. Define Features (X) and Target (y)
X = df.drop(columns=[df.columns[0], 'disease_label']).values
y = df['disease_label'].values

In [7]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
def build_model(hp):
    model = Sequential()
    model.add(Input(shape=(X.shape[1],)))
    
    # Tune the number of layers (between 2 and 4)
    for i in range(hp.Int('num_layers', 2, 4)):
        model.add(layers.Dense(
            units=hp.Int(f'units_{i}', min_value=256, max_value=1024, step=128),
            activation='relu',
            kernel_regularizer=regularizers.l2(hp.Float('l2', 1e-5, 1e-3, sampling='log'))
        ))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(hp.Float(f'dropout_{i}', 0.2, 0.5, step=0.1)))
    
    model.add(layers.Dense(len(label_encoder.classes_), activation='softmax'))
    
    # Tune the learning rate
    lr = hp.Float('learning_rate', 1e-4, 1e-2, sampling='log')
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [9]:
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=20,
    factor=3,
    directory='keras_tuner_dir',
    project_name='disease_prediction',
    overwrite=False
)

Reloading Tuner from keras_tuner_dir\disease_prediction\tuner0.json


In [10]:
# 1. Keep your existing callback definitions
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=2, verbose=1, min_lr=1e-6
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
)

In [11]:
tuner.search(
    X_train, y_train,
    epochs=50,             # Set high; EarlyStopping will cut it short per trial
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[lr_scheduler, early_stop] # Use your existing callbacks here
)

# Get the best parameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"The search is complete. The optimal number of layers is {best_hps.get('num_layers')}.")

Trial 30 Complete [00h 40m 00s]
val_accuracy: 0.8592804074287415

Best val_accuracy So Far: 0.8607786893844604
Total elapsed time: 15h 45m 22s
The search is complete. The optimal number of layers is 2.


In [12]:
# 1. Get the best hyperparameters discovered by the tuner
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# 2. Build the model with those specific settings
final_model = tuner.hypermodel.build(best_hps)

In [13]:
# 3. Train it one last time on the full dataset for more epochs
history = final_model.fit(
    X_train, y_train,
    epochs=100, # Give it plenty of room to reach 90%
    validation_data=(X_test, y_test),
    callbacks=[lr_scheduler, early_stop], # Keep your refined callbacks
    batch_size=512 # Faster training
)

Epoch 1/100
386/386 ━━━━━━━━━━━━━━━━━━━━ 28s 68ms/step - accuracy: 0.4301 - loss: 3.8629 - val_accuracy: 0.6834 - val_loss: 3.8137 - learning_rate: 1.3370e-04
Epoch 2/100
386/386 ━━━━━━━━━━━━━━━━━━━━ 44s 74ms/step - accuracy: 0.7514 - loss: 1.4103 - val_accuracy: 0.8195 - val_loss: 0.8831 - learning_rate: 1.3370e-04
Epoch 3/100
386/386 ━━━━━━━━━━━━━━━━━━━━ 26s 68ms/step - accuracy: 0.8103 - loss: 0.8631 - val_accuracy: 0.8433 - val_loss: 0.5964 - learning_rate: 1.3370e-04
Epoch 4/100
386/386 ━━━━━━━━━━━━━━━━━━━━ 46s 82ms/step - accuracy: 0.8325 - loss: 0.6668 - val_accuracy: 0.8526 - val_loss: 0.4994 - learning_rate: 1.3370e-04
Epoch 5/100
386/386 ━━━━━━━━━━━━━━━━━━━━ 41s 82ms/step - accuracy: 0.8436 - loss: 0.5724 - val_accuracy: 0.8583 - val_loss: 0.4517 - learning_rate: 1.3370e-04
Epoch 6/100
386/386 ━━━━━━━━━━━━━━━━━━━━ 41s 81ms/step - accuracy: 0.8486 - loss: 0.5216 - val_accuracy: 0.8581 - val_loss: 0.4238 - learning_rate: 1.3370e-04
Epoch 7/100
386/386 ━━━━━━━━━━━━━━━━━━━━ 39s 7

In [14]:

# 4. Save this 'Champion' version
final_model.save('disease_ann_model__keras_tuner_8606.keras')